# ISOM 835 · The Churn Detective
**A case file in seven parts — and a companion to Homework 1**

Last year 1,869 customers walked out of a telecom company. Nobody knows exactly why. You are the new analyst, and this notebook is your case file: 7,043 customers, 21 columns, one question from the CFO — *who is about to leave, and what should we do about it?*

Seven case files. Each one teaches a pandas or scikit-learn move, asks you to make a guess before you run the cell (that is where the fun is — your instincts vs. the data), and ends with a check cell that prints ✓ when you have it. The seven files line up with the six parts of Homework 1, so when you finish the case you have finished most of the homework.

**How to use it.** File → Save a copy in Drive first. Run the cells in order. Cells marked `# YOUR CODE` have `None` placeholders — replace them. Cells marked `# check` tell you whether you got it. Budget 60–90 minutes. Break things on purpose; the undo button is free.

In [ ]:
# Environment check — run this first
import sys, pandas as pd, numpy as np, sklearn
print('Python', sys.version.split()[0], '· pandas', pd.__version__, '· scikit-learn', sklearn.__version__)

PASSED = set()   # the checks you clear go here; the scorecard at the end reads it

def guess(label, my_guess, actual):
    """Compare your instinct with the data. Guesses are rates between 0 and 1."""
    if my_guess is None:
        print(f'→ Type a number for my_guess first, then re-run. (The answer is hiding until you do.)'); return
    gap = abs(my_guess - actual) * 100
    verdict = ('🎯 within 2 points — you have a feel for this data' if gap <= 2 else
               '👍 within 10 points' if gap <= 10 else
               '🤔 more than 10 points off — and that is the interesting part: why?')
    print(f'{label}: you guessed {my_guess:.0%}, the data says {actual:.1%}. {verdict}')

## Case file 0 — Open the evidence

Every investigation starts the same way: how big is the file, what is in it, and what is the base rate of the thing you are trying to predict.

**Guess before you run:** out of 7,043 customers, what share left? Type your guess as a fraction (0.30 means 30%).

In [ ]:
url = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/telco_churn.csv'
df = pd.read_csv(url)
df.shape

In [ ]:
df.head()

In [ ]:
df.info()   # 21 columns. Read the Dtype column carefully — one of them is lying.

In [ ]:
my_guess = None      # your guess for the churn rate, e.g. 0.30

churn_rate = (df['Churn'] == 'Yes').mean()
guess('Churn rate', my_guess, churn_rate)

That number is your **baseline**. A "model" that says *nobody churns* is right 73.5% of the time. Every model you build this semester has to beat the baseline, or it is a decoration.

The check below asks for the churn rate as a variable and a 0/1 column called `churned`, which you will use all night: the mean of a 0/1 column is a rate.

In [ ]:
# YOUR CODE
churned_col = None          # (df['Churn'] == 'Yes').astype(int) — a 0/1 column
baseline_accuracy = None    # the accuracy of always predicting 'No'  (hint: 1 - churn_rate)

if churned_col is not None:
    df['churned'] = churned_col
    print(df['churned'].mean(), baseline_accuracy)

In [ ]:
# check 0
assert 'churned' in df.columns and set(df['churned'].unique()) == {0, 1}, 'churned should be a 0/1 column on df'
assert round(df['churned'].mean(), 3) == 0.265, f'churned.mean() should be ≈ 0.265, got {df["churned"].mean()}'
assert baseline_accuracy is not None and round(baseline_accuracy, 3) == 0.735, f'baseline_accuracy should be ≈ 0.735, got {baseline_accuracy}'
PASSED.add(0); print('✓ Case 0 closed — baseline 73.5%. Beat it or go home.')

## Case file 1 — The tampered column  *(HW1 part 1)*

`.info()` said `TotalCharges` is `object` — text — in a table where it should obviously be a number. Somewhere in 7,043 rows something is not a number. Find it.

`pd.to_numeric(..., errors='coerce')` converts what it can and turns the rest into `NaN`. Then `isnull()` points straight at the rows that failed.

In [ ]:
df['TotalCharges'].head()

In [ ]:
# YOUR CODE
total_numeric = None   # pd.to_numeric(df['TotalCharges'], errors='coerce')
n_bad = None           # how many rows failed to convert?  (.isnull().sum())
print(n_bad)

In [ ]:
# Look at the culprits before you decide what to do with them
if total_numeric is not None:
    display(df.loc[total_numeric.isnull(), ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']])
else:
    print('→ Fill in total_numeric in the cell above first.')

All eleven have `tenure` 0 — customers so new they have not been billed yet. The text field is a single space. So this is not corruption, it is *a new customer*, and the honest total is **0**.

That is the sentence HW1 part 1 asks for. Notice the shape of the argument: *look at the rows, understand why they are missing, then choose*. Never `fillna` blind.

In [ ]:
# YOUR CODE — the numeric column with the eleven filled with 0
total_fixed = None    # total_numeric.fillna(0)

if total_fixed is not None:
    df['TotalCharges'] = total_fixed
    print(df['TotalCharges'].dtype, df['TotalCharges'].isnull().sum())

In [ ]:
# check 1
assert n_bad == 11, f'eleven rows fail to convert, you found {n_bad}'
assert pd.api.types.is_numeric_dtype(df['TotalCharges']), 'TotalCharges should be numeric now'
assert df['TotalCharges'].isnull().sum() == 0, 'no missing TotalCharges should remain'
assert (df.loc[df['tenure'] == 0, 'TotalCharges'] == 0).all(), 'the tenure-0 customers should have TotalCharges 0'
PASSED.add(1); print('✓ Case 1 closed — the column was not corrupt, it was new customers.')

## Case file 2 — Interrogate the suspects  *(HW1 part 2)*

A `groupby` is an interrogation: line up every value of a column, ask each group the same question. With a 0/1 target, `groupby(col)['churned'].mean()` is the churn rate per group, and `.agg(['mean', 'count'])` keeps you honest about small groups.

**Guess before you run:** which contract type churns most, and roughly what share? Month-to-month, one year, or two year?

In [ ]:
my_guess = None      # your guess for the month-to-month churn rate

by_contract = df.groupby('Contract')['churned'].agg(['mean', 'count']).sort_values('mean', ascending=False)
guess('Month-to-month churn', my_guess, by_contract.loc['Month-to-month', 'mean'])

In [ ]:
by_contract     # the full interrogation: rate and head-count per contract type

One sentence of interpretation, the kind HW1 wants: *Month-to-month customers churn at 43%, fifteen times the rate of two-year customers — the contract is not a detail, it is the exit door.*

A helper for the rest of the night. `interrogate(col)` prints the table and draws it, so you can try a column in one line.

In [ ]:
import matplotlib.pyplot as plt

def interrogate(col, min_count=30):
    """Churn rate and count for every value of col, sorted, with a bar chart."""
    assert 'churned' in df.columns, 'Case 0 first: df needs the 0/1 churned column'
    t = df.groupby(col, observed=True)['churned'].agg(['mean', 'count'])
    t = t[t['count'] >= min_count].sort_values('mean', ascending=False)
    ax = t['mean'].plot(kind='barh', color='#2ee6c5', figsize=(6, 0.4 * len(t) + 1))
    ax.axvline(df['churned'].mean(), color='#ff6b8b', linestyle='--', label='overall 26.5%')
    ax.set_xlabel('churn rate'); ax.invert_yaxis(); ax.legend(loc='lower right'); ax.set_title(f'Churn by {col}')
    plt.tight_layout(); plt.show()
    return t

interrogate('Contract')

Now two more suspects, your choice. HW1 suggests `InternetService` and `PaymentMethod`; `TechSupport`, `SeniorCitizen`, `PaperlessBilling` are also good. Store the two tables as `table_2` and `table_3`.

In [ ]:
# YOUR CODE
table_2 = None    # interrogate('InternetService')
table_3 = None    # interrogate('PaymentMethod')

In [ ]:
# check 2
for name, t in [('table_2', table_2), ('table_3', table_3)]:
    assert isinstance(t, pd.DataFrame) and 'mean' in t.columns, f'{name} should be the DataFrame returned by interrogate(...)'
    assert t['mean'].max() > df['churned'].mean(), f'{name}: at least one group should churn above the overall rate — try another column'
PASSED.add(2); print('✓ Case 2 closed — three interrogations on the record. Now write one sentence under each in your HW notebook.')

**Two suspects at once.** Real patterns are often interactions. `groupby` two columns, then `unstack()` to lay them out as a grid. Where is the worst cell?

In [ ]:
df.groupby(['Contract', 'PaymentMethod'])['churned'].mean().unstack().round(3)

In [ ]:
pd.crosstab(df['Contract'], df['Churn'])     # counts, not rates — the business cares about both

## Case file 3 — Invent a clue  *(HW1 part 3)*

The columns you were given are the evidence. The columns you *build* are the clues. Each new column is a hypothesis about why people leave, written as arithmetic.

Worked example: **how many add-on services does the customer have?** Hypothesis: the more of our services you use, the harder it is to leave. Six columns say "Yes"/"No"; compare each to "Yes" and sum across the row (`axis=1`).

In [ ]:
addons = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['n_addons'] = (df[addons] == 'Yes').sum(axis=1)
interrogate('n_addons')

Not what the hypothesis said. Zero add-ons churn at 21%, **one** add-on churns at 46%, and only from there does the rate fall. The single-add-on customers are the ones to look at — why would one extra service make you *more* likely to leave? (Hint: which add-on is it usually, and what kind of internet do those customers have?)

Now yours. **Average charge per month of tenure** — hypothesis: customers whose bills crept up over time feel it. `TotalCharges / tenure`, but tenure can be 0, so replace 0 with 1 first. Then a **new-customer flag**: `tenure <= 6`.

In [ ]:
# YOUR CODE
charges_per_month = None     # df['TotalCharges'] / df['tenure'].replace(0, 1)
is_new = None                # (df['tenure'] <= 6).astype(int)

if charges_per_month is not None and is_new is not None:
    df['charges_per_month'] = charges_per_month
    df['is_new'] = is_new
    display(df[['tenure', 'MonthlyCharges', 'TotalCharges', 'charges_per_month', 'is_new']].head())

In [ ]:
# check 3
assert 'charges_per_month' in df.columns and 'is_new' in df.columns, 'fill in charges_per_month and is_new in the cell above'
assert df['charges_per_month'].notnull().all() and np.isfinite(df['charges_per_month']).all(), 'charges_per_month has NaN or inf — did you replace tenure 0?'
assert abs(df['charges_per_month'].iloc[1] - 1889.5 / 34) < 0.01, 'row 1 should be 1889.5 / 34 ≈ 55.57'
assert set(df['is_new'].unique()) == {0, 1} and df['is_new'].sum() == df['tenure'].le(6).sum(), 'is_new should be 1 exactly when tenure <= 6'
PASSED.add(3); print('✓ Case 3 closed — three clues built: n_addons, charges_per_month, is_new.')

Does the clue carry signal? For a number, compare its average between churners and stayers. For a flag, interrogate it.

In [ ]:
df.groupby('churned')[['tenure', 'MonthlyCharges', 'charges_per_month', 'n_addons']].mean().round(2)

In [ ]:
interrogate('is_new')

**Your own clue.** Build one more column that is *your* hypothesis — a ratio, a flag, a count, an interaction (for example `Contract == 'Month-to-month'` and `PaymentMethod == 'Electronic check'` together). Name it whatever you like, and store its name in `my_feature`. Then interrogate it or compare group means. Write the hypothesis as a comment: HW1 asks you to explain why it might carry signal.

In [ ]:
# YOUR CODE — hypothesis: ...
my_feature = None      # e.g. 'mtm_echeck'
# df[my_feature] = ...

## Case file 4 — The time machine  *(HW1 part 4)*

The most famous way to build a useless model: use information you would not have had at the moment of prediction.

Suppose the prediction is made **on the day a customer signs up**, not today. Walk down the column list and ask of each one: *would I know this on day one?* Anything that accumulates over the relationship — months as a customer, money paid so far — did not exist yet. Anything the customer chose at sign-up — contract, payment method, the plan price — did.

Put the names of every column you would have to **drop** into the list below. Include the columns you built tonight if they depend on time.

In [ ]:
list(df.columns)

In [ ]:
# YOUR CODE — column names you would NOT know on sign-up day
unknown_at_signup = [
    # 'tenure',
]

In [ ]:
# check 4
must_drop = {'tenure', 'TotalCharges', 'charges_per_month', 'is_new'}
known_day_one = {'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'Contract', 'PaymentMethod', 'MonthlyCharges', 'PaperlessBilling'}
chosen = set(unknown_at_signup)
missing = must_drop - chosen
wrong = chosen & known_day_one
assert not missing, f'These accumulate over time and did not exist on day one: {sorted(missing)}'
assert not wrong, f'These are chosen at sign-up, so you would know them: {sorted(wrong)}'
PASSED.add(4); print('✓ Case 4 closed. Debatable ones (add-on services, MultipleLines) can go either way — say which way and why in your HW.')

Notice what you just lost: `tenure` is the single strongest predictor in this table, and at sign-up it does not exist. That is why a churn model built for *existing* customers and a churn model built for *new* customers are two different models. The Leakage Time Machine on the Session 2 page lets you drag that moment around and watch a real model's score move.

## Case file 5 — Prove it with a model  *(HW1 part 5)*

Interrogations find suspects one at a time. A model weighs all of them at once. Same recipe as Session 1: split, fit, predict, compare with the baseline. The helper below does the whole thing for any list of feature columns, so you can run experiments in one line.

Text columns are turned into 0/1 columns with `pd.get_dummies` — one column per category. (Next week the Pipeline does this properly; tonight this is enough.)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

def fit_and_report(feature_cols, show_matrix=True):
    """Logistic regression on feature_cols; returns test accuracy and prints the confusion matrix."""
    X = pd.get_dummies(df[feature_cols], drop_first=True, dtype=int)
    y = df['churned']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=835)
    model = LogisticRegression(max_iter=2000).fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    base = 1 - y_test.mean()
    print(f'features: {len(feature_cols)} columns → {X.shape[1]} after dummies')
    print(f'accuracy {acc:.3f}   baseline (always No) {base:.3f}   lift {acc - base:+.3f}')
    if show_matrix:
        cm = pd.DataFrame(confusion_matrix(y_test, pred), index=['actual: stayed', 'actual: churned'],
                          columns=['predicted: stayed', 'predicted: churned'])
        print(cm)
    return acc

acc_session1 = fit_and_report(['tenure', 'MonthlyCharges', 'Contract'])

Read the confusion matrix like a manager. The bottom row is the churners: how many did the model catch (bottom-right) and how many slipped past it (bottom-left)? Accuracy alone hides that row.

**Your experiment.** Add your engineered features and any columns you like to `my_features`, run, and beat the Session 1 score. Keep the *best* list. (Do not add `Churn` or `churned` — that is the answer key, and the time machine from Case 4 applies to your HW answer too.)

In [ ]:
# YOUR CODE
my_features = ['tenure', 'MonthlyCharges', 'Contract',
               # 'n_addons', 'charges_per_month', 'is_new', 'InternetService', 'PaymentMethod', ...
              ]
acc_mine = fit_and_report(my_features)

In [ ]:
# check 5
assert 'Churn' not in my_features and 'churned' not in my_features and 'customerID' not in my_features, 'no answer key, no ID column'
assert acc_mine > 0.735, f'accuracy {acc_mine:.3f} does not beat the baseline 0.735'
assert acc_mine >= acc_session1, f'accuracy {acc_mine:.3f} is below the Session 1 model ({acc_session1:.3f}) — add the engineered features'
PASSED.add(5); print(f'✓ Case 5 closed — {acc_mine:.1%} vs baseline 73.5%. Report accuracy, the matrix, and the baseline in your HW.')

**Why accuracy lies, in one cell.** The model above gets about 80%. A model that flags *nobody* gets 73.5%. So the honest measure of the work is the seven points between them — and, for a retention team, the bottom row of the matrix. Session 6 turns that row into dollars.

In [ ]:
# The gap that matters: how many of the churners in the test set did each model catch?
for cols in [['tenure'], ['tenure', 'Contract'], my_features]:
    print(f'--- {cols if len(cols) < 4 else str(len(cols)) + " features"}')
    fit_and_report(cols, show_matrix=True); print()

## Case file 6 — Your own case  *(HW1 part 6)*

Every business question about the future is a prediction problem, once you name four things:

| | Telco churn | A hospital | A retailer |
|---|---|---|---|
| **Unit** — one row is one… | customer | patient discharge | SKU-store-week |
| **Target** — the column we want | churned in the next 30 days (0/1) | readmitted within 30 days (0/1) | units sold (number) |
| **Horizon** — predicted when, about when | at month-end, for next month | on discharge day, for the next 30 days | Monday, for the coming week |
| **Decision** — what changes | who the retention team calls | who gets a follow-up call | how much to reorder |

Frame one from your own work or life. The check only looks that each field is filled with a real sentence — the judgment is yours, and this may become your final project.

In [ ]:
# YOUR CODE
my_case = {
    'unit':     None,   # 'one row is one ...'
    'target':   None,   # 'the column we want: ... (0/1 or a number)'
    'horizon':  None,   # 'predicted at ..., about ...'
    'decision': None,   # 'what changes if the prediction is high: ...'
}

In [ ]:
# check 6
for k in ['unit', 'target', 'horizon', 'decision']:
    v = my_case.get(k)
    assert isinstance(v, str) and len(v.strip()) >= 15, f'{k!r} needs a real sentence, not {v!r}'
PASSED.add(6); print('✓ Case 6 closed — paste these four lines into the final markdown cell of your HW notebook.')

## Scorecard

In [ ]:
ranks = {0: 'Civilian', 1: 'Civilian', 2: 'Trainee', 3: 'Rookie', 4: 'Inspector', 5: 'Detective', 6: 'Chief Inspector', 7: 'Chief of Analytics'}
n = len(PASSED)
print('Cases closed:', ' '.join(('✓' if i in PASSED else '·') + str(i) for i in range(7)))
print(f'{n} of 7 → rank: {ranks[n]}')
if n < 7: print('Still open:', sorted(set(range(7)) - PASSED))

## Your turn — cold cases

No checks here. These are the questions a curious analyst asks next, and any of them makes a good paragraph in your HW.

1. **The single-add-on mystery.** Among customers with exactly one add-on, which add-on is it most often, and what internet service do they have? Does that explain the 46%?
2. **Fiber optic.** Fiber customers churn at 42%. Is it the product, or the *price*? Compare `MonthlyCharges` for fiber vs DSL, then churn by `InternetService` *within* a narrow price band.
3. **Who do you call first?** The retention team can call 300 customers. Use `df.sort_values` on a segment table (Contract × PaymentMethod × is_new) to name the segment, then count how many customers are in it. Is 300 the right number?
4. **The time machine, again.** Rerun `fit_and_report` with only the columns you *would* know at sign-up. How much accuracy does honesty cost? Is a sign-up-day model still worth building?

In [ ]:
# Your turn — work here

## Case notes

- `pd.to_numeric(errors='coerce')` + `isnull()` finds bad values; *look at the rows* before you fill them.
- `groupby(col)['churned'].agg(['mean', 'count'])` is an interrogation; `unstack()` and `crosstab` interrogate two suspects at once.
- A new column is a hypothesis. Test it with a group mean before you trust it.
- Drop what you would not know at prediction time. `tenure` at sign-up is the classic leak.
- Accuracy is only meaningful next to the baseline and the confusion matrix.
- A prediction problem is a unit, a target, a horizon, and a decision.

*Next: the Session 3 notebook turns tonight's hand-built steps into a scikit-learn Pipeline so they are fit on training rows only.*